# Module 1 — Python essentials

**Who this is for:** you have never programmed in Python, or it has been a while.

> **Self-check — skip this module if you can already:**
>
> - [ ] run a notebook cell and read what it printed
> - [ ] explain the difference between a list and a dict
> - [ ] write a function with a `for` loop and an `if`
> - [ ] read a Python error message and find the line that caused it
>
> If you can tick every box, go to the next module. If not, work through the parts you can't tick. Before
> jumping to a section, run the module's **setup cell** (the first code cell): later sections use variables
> defined earlier, so run the cells above a section if you get a `NameError`.

**You will learn**
- how notebooks run, and why order matters
- variables, numbers and text
- lists, dicts, loops and conditions
- writing your own functions
- importing libraries
- reading error messages and getting help

**Where the course uses this:** every demo and lab notebook. Nothing in the course needs more Python than this module plus Module 2.

*Run each cell with **Shift+Enter**, top to bottom. Nothing here needs data files; once the libraries are
installed (see the [overview](README.md#setup-first)), the notebook runs offline.*

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/farhad-abtahi/CM2013/blob/main/python_tutorial/01_python_essentials.ipynb)
&nbsp; Part of the **Python for biosignal processing** tutorial — see the [overview](README.md) to choose which modules you need.

## 1. Notebooks: cells run in the order *you* run them

A notebook is a list of cells. **Code cells** run Python; **markdown cells** (like this one) are text.
Python remembers everything you have run so far, in the order you ran it — not the order on the page.
If a notebook behaves strangely, use **Runtime → Restart and run all** (Colab) or **Restart → Run All**
(VS Code / Jupyter) to start clean.

In [ ]:
# Lines starting with # are comments: Python ignores them.
print("Hello, biosignals!")
2 + 3          # the value of the LAST line of a cell is shown automatically

## 2. Variables, numbers and text

A variable is a name for a value. Python works out the type for you.

In [ ]:
fs = 250                    # an int: sampling rate in Hz
duration_s = 10.0           # a float: seconds
subject = "S01"             # a str (text)
is_clean = True             # a bool: True / False

n_samples = int(fs * duration_s)
print(type(fs), type(duration_s), type(subject), type(is_clean))
print(f"{subject}: {n_samples} samples at {fs} Hz")   # an f-string puts values inside text

**Watch out:** `/` always gives a float (`7 / 2` is `3.5`); `//` is *floor* division, which rounds down
(`7 // 2` is `3`, and `-7 // 2` is `-4`, not `-3`); `**` is the power (`2 ** 10` is `1024`; `^` is *not* a
power in Python).

In [ ]:
print(7 / 2, 7 // 2, -7 // 2, 7 % 2, 2 ** 10)

## 3. Lists and dicts

A **list** is an ordered sequence. Indexing starts at **0**, and negative indices count from the end.
A **dict** maps names (keys) to values — handy for settings and for metadata about a recording.

In [ ]:
rr_s = [0.82, 0.80, 0.85, 0.79, 0.81]      # RR intervals in seconds
print(rr_s[0], rr_s[-1])                  # first and last
print(rr_s[1:3])                          # a slice: items 1 and 2 (the end index is NOT included)
rr_s.append(0.83)
print(len(rr_s), "intervals")

recording = {"subject": "S01", "fs": 250, "channels": ["ECG I", "ECG II"], "units": "mV"}
print(recording["fs"], recording["channels"][1])
recording["notes"] = "electrode re-attached at 12 min"
print(list(recording.keys()))

## 4. Loops and conditions

A `for` loop repeats a block for each item. The block is defined by **indentation** (4 spaces) —
in Python indentation is not decoration, it is the syntax.

In [ ]:
# The thresholds below are toy categories for practising if/elif/else, not clinical definitions.
for i, rr in enumerate(rr_s):
    hr = 60 / rr
    if hr > 74:
        label = "fast"
    elif hr < 71:
        label = "slow"
    else:
        label = "normal"
    print(f"beat {i}: RR={rr:.2f} s  HR={hr:5.1f} bpm  {label}")

A **list comprehension** builds a list in one line — you will see these often in the course notebooks:

In [ ]:
hr_bpm = [60 / rr for rr in rr_s]
fast = [hr for hr in hr_bpm if hr > 74]
print([round(h, 1) for h in hr_bpm], "fast:", [round(h, 1) for h in fast])

## 5. Functions

Wrap anything you do more than once in a function. Give it a clear name, document what goes in and
what comes out, and `return` the result.

In [ ]:
def mean_instantaneous_hr(rr_intervals_s):
    """Average of the beat-by-beat heart rates, mean(60 / RR), in beats per minute.

    rr_intervals_s: list of RR intervals in seconds; must be non-empty, finite and positive.
    """
    if len(rr_intervals_s) == 0:
        raise ValueError("need at least one RR interval")
    total = 0.0
    for rr in rr_intervals_s:
        if not 0 < rr < float("inf"):       # also False for NaN, so NaN is rejected too
            raise ValueError(f"RR interval must be a finite positive number, got {rr}")
        total += 60 / rr
    return total / len(rr_intervals_s)

print(f"{mean_instantaneous_hr(rr_s):.1f} bpm")

# A quick self-check. If this line raises an error, the function is wrong.
assert abs(mean_instantaneous_hr([1.0, 1.0]) - 60.0) < 1e-9

`assert condition` does nothing when the condition is true and stops with an error when it is false.
The labs use asserts as **sanity checks** — the same habit is worth keeping in your capstone.

**Name what you compute.** "Mean heart rate" can mean two different things, and they differ when the rhythm
is irregular:

In [ ]:
rr_irregular = [0.5, 1.5]                       # one short and one long interval: 2 s, 2 beats
print("mean of beat-by-beat rates, mean(60/RR):", mean_instantaneous_hr(rr_irregular), "bpm")
print("beats per time, 60 / mean(RR):          ", 60 / (sum(rr_irregular) / len(rr_irregular)), "bpm")

The second one, 60 / mean(RR), is "beats per minute over the recording". Module 4 uses it. Neither is
wrong; say which one you report.

## 6. Importing libraries

Python's power for signal processing comes from libraries. You import them once at the top of a notebook,
usually with a short conventional name:

In [ ]:
import numpy as np                  # arrays and maths
import matplotlib.pyplot as plt     # plotting
from scipy import signal            # filters, spectra, peaks

print(np.__version__)

## 7. Reading errors

Errors are normal. Read them **from the bottom up**: the last line says *what* went wrong, and the lines
above it point to *where*. The cell below triggers an error on purpose and prints the full error report
(the *traceback*), so the notebook still runs to the end:

In [ ]:
import traceback
rr_bad = [0.8, 0.0, 0.9]
try:
    mean_instantaneous_hr(rr_bad)
except ValueError:
    traceback.print_exc()          # the same report you would see if the error were not caught

Read it from the bottom: the last line says `ValueError: RR interval must be a finite positive number, got 0.0`. The lines
above show the chain of calls, ending in the line of `mean_instantaneous_hr` that raised it. The fix belongs in
the *data* (an RR interval of 0 s is impossible), not in the function.

| Error | Usual cause |
|---|---|
| `NameError: name 'x' is not defined` | You didn't run the cell that defines `x`, or there is a typo |
| `IndentationError` | Mixed or missing indentation inside a loop, `if` or function |
| `IndexError: list index out of range` | Asking for item `n` of a list with only `n` items (indices go to `n-1`) |
| `ModuleNotFoundError` | The library is not installed in this environment (see Module 6) |
| `TypeError` | Wrong kind of value, e.g. adding text to a number |
| `ValueError: operands could not be broadcast` | Two arrays have shapes that don't fit together (see Module 2) |

**Getting help:** `help(np.mean)`, or put `?` after a name in a notebook (`np.mean?`). Searching the exact
last line of an error message usually finds the answer.

## Try it yourself

Write `heart_rate_variability(rr_intervals_s)` that returns the **population standard deviation of the RR
intervals in milliseconds**: the square root of the mean squared difference from the mean (divide by N).
Use a loop, no libraries. Then run the check cell below it.

*Background:* in heart-rate-variability work this measure is called **SDNN** when it is computed on cleaned
normal-to-normal (NN) intervals, and many tools divide by N − 1 instead of N (the *sample* standard deviation).
For two intervals the choice matters: 100 ms vs 141 ms. Always state which convention you use.

In [ ]:
def heart_rate_variability(rr_intervals_s):
    # TODO: compute the mean, then the mean squared difference from it (divide by N), then the square
    #       root (** 0.5). Remember to convert seconds to milliseconds.
    ...

In [ ]:
# Check: keep this cell separate, so it survives when you paste in a solution.
result = heart_rate_variability([0.8, 1.0])
if result is None:
    print("Not done yet: heart_rate_variability returns None.")
else:
    print("correct" if abs(result - 100.0) < 1e-6 else f"not yet: got {result}, expected 100.0")

**Solution:** try it yourself first, then compare with the [solutions notebook](solutions.ipynb) (Module 1). Yours may differ and still be right.

**Next:** Module 2 turns lists into NumPy arrays, which is how every signal in the course is stored.